# Unsharp masking and GGM for the PSZ cluster

A range of unsharp masked and GGM filtered images are produced using a series of kernels. These images are used to find the location of putative edges. 

## Imports

In [ ]:
import sys
sys.path.append('./utils')

# Third party libraries
from astropy import units as u
from astropy.coordinates import SkyCoord

%matplotlib widget

from astropy.cosmology import FlatLambdaCDM

from astropy.visualization import quantity_support
quantity_support()

# Local packages
from utils.utils_unsharp_masking_GGM import (
    make_unsharp_masked_images, 
    plot_unsharp_masked_images,
    make_GGM_images, 
    plot_ggm_images, 
    PSZ_ggm_um_figure,
    Kernel, 
    UM)
from utils.utils_img_preproc import get_hdr_img_wcs, beautify_image, slice_radio, slice_Xray

## Constants

Set cosmology

In [ ]:
cosmo = FlatLambdaCDM(H0=69.6, Om0=0.286, Tcmb0=2.725)

Cluster properties

In [ ]:
# Cluster name
target_name = "PSZ"
# Cluster redshift
z = 0.234
# Cluster coordinates
ref_coords = SkyCoord(144.85407041666667 * u.deg, +40.786631111111106 * u.deg)

Path to X-ray and radio data

In [ ]:
XMM_file = "./input_data/xmm_flux_full_nopsrc_NaN.fits"
Chandra_file = "./input_data/chandra_flux_nopsrc_NaN.fits"
radio_file = "./input_data/pszg181.06+48.47_maskROBUST-0.5-MFS-image_smooth_zoom.fits"
rms_radio = 2 * 10**-4
size = (15.0 * u.arcmin, 14.0 * u.arcmin)  # FOV for cluster
size_small = (7.5 * u.arcmin, 7 * u.arcmin)  # Smaller FOV for doing exploratory work

Read in *XMM-Newton* and *Chandra* data.

In [ ]:
XMM_img, XMM_hdr, _ = get_hdr_img_wcs(XMM_file)
Chandra_img, Chandra_hdr, _ = get_hdr_img_wcs(Chandra_file)

## Unsharp masking

Best unsharp masked images for `PSZ` are as follow:
* XMM-Newton: 2, 4
* Chandra: 4, 49

### XMM unsharp masking

In [ ]:
UM_XMM_prop = UM(s1=Kernel(min=1, max=9, step=1), s2=Kernel(min=2, max=18, step=1))
make_unsharp_masked_images(
    XMM_img, XMM_hdr, target_name, "./unsharp_masked_images/", UM_XMM_prop
)

In [ ]:
UM_XMM_plot = UM(s1=Kernel(min=2, max=4, step=1), s2=Kernel(min=4, max=10, step=2))
plot_unsharp_masked_images(
    XMM_img,
    XMM_hdr,
    target_name,
    "./unsharp_masked_images/",
    ref_coords,
    size_small,
    UM_XMM_plot,
)

### Chandra unsharp masking

In [ ]:
UM_Chandra_prop = UM(
    s1=Kernel(min=11, max=12, step=1), s2=Kernel(min=49, max=50, step=5)
)
make_unsharp_masked_images(
    Chandra_img, Chandra_hdr, target_name, "./unsharp_masked_images/", UM_Chandra_prop
)

In [ ]:
# UM_Chandra_plot = UM(s1=Kernel(min=4, max=12, step=2), s2=Kernel(min=4, max=50, step=5))
UM_Chandra_plot = UM(s1=Kernel(min=4, max=5, step=1), s2=Kernel(min=9, max=34, step=5))

plot_unsharp_masked_images(
    Chandra_img,
    Chandra_hdr,
    target_name,
    "./unsharp_masked_images/",
    ref_coords,
    size_small,
    UM_Chandra_plot,
    vmin=-1e-9,
    vmax=5e-9,
)

## GGM

### XMM GGM

In [ ]:
XMM_proc = beautify_image(XMM_img.byteswap().view(XMM_img.dtype.newbyteorder()))
make_GGM_images(XMM_proc, XMM_hdr, target_name, "./ggm_images/")

In [ ]:
plot_ggm_images(XMM_img, XMM_hdr, target_name, "./ggm_images/")

### Chandra GGM 

In [ ]:
Chandra_proc = beautify_image(
    Chandra_img.byteswap().view(Chandra_img.dtype.newbyteorder())
)
Chandra_kernel = Kernel(min=1, max=25, step=1)
make_GGM_images(Chandra_proc, Chandra_hdr, target_name, "./ggm_images/", Chandra_kernel)

In [ ]:
Chandra_kernel = Kernel(min=10, max=25, step=1)
plot_ggm_images(
    Chandra_img,
    Chandra_hdr,
    target_name,
    "./ggm_images/",
    Chandra_kernel,
    vmin=0,
    vmax=5e-10,
)

## Compare XMM | Chandra Unsharp masking vs GGM

In [ ]:
images = [
    "./unsharp_masked_images/PSZ_XMM_unsharp_masking_02_04.fits",
    "./unsharp_masked_images/PSZ_CHANDRA_unsharp_masking_04_49.fits",
    "./unsharp_masked_images/PSZ_CHANDRA_unsharp_masking_11_49.fits",
    "./ggm_images/PSZ_XMM_GGM_03.fits",
    "./ggm_images/PSZ_CHANDRA_GGM_10.fits",
    "./ggm_images/PSZ_CHANDRA_GGM_20.fits",
]
images_zoom = [slice_Xray(im, ref_coords, size) for im in images]
radio_path = slice_radio(radio_file, ref_coords, size)

regions = {
    "N edge": "./input_data/SB_N_sector.reg",
    "NW edge": "./input_data/SB_NW_sector.reg",
    "S edge": "./input_data/SB_S_sector.reg",
}
bays = "./input_data/Bays.reg"
bay_labels = ["Bay + loop?", "Bay?", "Clump"]

PSZ_ggm_um_figure(
    images_zoom, radio_path, rms_radio, z, cosmo, regions, bays, bay_labels
)